# Tesla Deliveries — End-to-End ML Pipeline
**Dataset:** `tesla_deliveries_dataset_2015_2025.csv`  
**Covers:** Preprocessing → EDA → Feature Engineering → Regression Modeling → Hyperparameter Tuning → Time Series Forecasting

## 0. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import lightgbm as lgb
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 4)})

RANDOM_STATE = 42
print('All libraries loaded successfully.')

## 1. Data Loading & Initial Inspection

In [ ]:
df = pd.read_csv('tesla_deliveries_dataset_2015_2025.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Preprocessing & Data Cleaning

In [ ]:
# ── Missing values
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values found.')

# ── Duplicates
print(f'\nDuplicate rows: {df.duplicated().sum()}')

# ── Categorical cardinality
print('\n=== Categorical Columns ===')
for col in df.select_dtypes(include='object').columns:
    print(f'  {col}: {df[col].nunique()} unique → {df[col].unique()}')

In [ ]:
# ── Outlier check via IQR on numeric columns
numeric_cols = ['Estimated_Deliveries', 'Production_Units', 'Avg_Price_USD',
                'Battery_Capacity_kWh', 'Range_km', 'CO2_Saved_tons', 'Charging_Stations']

outlier_report = {}
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    outlier_report[col] = n_out

outlier_df = pd.DataFrame.from_dict(outlier_report, orient='index', columns=['Outliers (IQR)'])
print(outlier_df)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── Distribution of key numeric features
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white')
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')
axes[-1].set_visible(False)
fig.suptitle('Distribution of Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Annual deliveries trend
annual = df.groupby('Year')['Estimated_Deliveries'].sum().reset_index()

plt.figure(figsize=(11, 4))
plt.plot(annual['Year'], annual['Estimated_Deliveries'], marker='o', linewidth=2, color='steelblue')
plt.fill_between(annual['Year'], annual['Estimated_Deliveries'], alpha=0.15, color='steelblue')
plt.title('Annual Global Tesla Deliveries (2015–2025)', fontsize=13, fontweight='bold')
plt.xlabel('Year'); plt.ylabel('Total Deliveries')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout(); plt.show()

In [ ]:
# ── Deliveries by Region and Model
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

region_del = df.groupby('Region')['Estimated_Deliveries'].sum().sort_values(ascending=True)
region_del.plot(kind='barh', ax=axes[0], color='#4C72B0')
axes[0].set_title('Total Deliveries by Region', fontweight='bold')
axes[0].set_xlabel('Deliveries')

model_del = df.groupby('Model')['Estimated_Deliveries'].sum().sort_values(ascending=True)
model_del.plot(kind='barh', ax=axes[1], color='#55A868')
axes[1].set_title('Total Deliveries by Model', fontweight='bold')
axes[1].set_xlabel('Deliveries')

plt.tight_layout(); plt.show()

In [ ]:
# ── Avg Price by Model
plt.figure(figsize=(9, 4))
price_model = df.groupby('Model')['Avg_Price_USD'].mean().sort_values(ascending=False)
bars = plt.bar(price_model.index, price_model.values, color=sns.color_palette('muted', len(price_model)))
plt.title('Average Price by Model (USD)', fontsize=13, fontweight='bold')
plt.ylabel('Avg Price (USD)')
for bar, val in zip(bars, price_model.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200, f'${val:,.0f}',
             ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ── Correlation heatmap
plt.figure(figsize=(10, 7))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Seasonality: monthly delivery patterns
monthly_avg = df.groupby('Month')['Estimated_Deliveries'].mean()
plt.figure(figsize=(10, 4))
plt.plot(monthly_avg.index, monthly_avg.values, marker='s', color='darkorange', linewidth=2)
plt.xticks(range(1, 13), ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.title('Average Monthly Deliveries (Seasonal Pattern)', fontsize=13, fontweight='bold')
plt.ylabel('Avg Deliveries'); plt.xlabel('Month')
plt.tight_layout(); plt.show()

In [ ]:
# ── Price vs Range scatter
plt.figure(figsize=(10, 5))
palette = sns.color_palette('tab10', n_colors=df['Model'].nunique())
for i, model in enumerate(df['Model'].unique()):
    sub = df[df['Model'] == model]
    plt.scatter(sub['Range_km'], sub['Avg_Price_USD'], label=model,
                alpha=0.5, s=18, color=palette[i])
plt.title('Avg Price vs Range by Model', fontsize=13, fontweight='bold')
plt.xlabel('Range (km)'); plt.ylabel('Avg Price (USD)')
plt.legend(title='Model', markerscale=1.5)
plt.tight_layout(); plt.show()

## 4. Feature Engineering

In [ ]:
# ── Create datetime index
df['Date'] = pd.to_datetime(df[['Year','Month']].assign(day=1))
df.sort_values('Date', inplace=True)
df.reset_index(drop=True, inplace=True)

# ── Cyclical month encoding
df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12)

# ── Time-based
df['Quarter']           = df['Month'].apply(lambda m: (m - 1) // 3 + 1)
df['Years_Since_Start'] = df['Year'] - 2015

# ── Interaction & ratio features
df['Price_Per_km']      = df['Avg_Price_USD'] / df['Range_km']
df['Util_Rate']         = df['Estimated_Deliveries'] / df['Production_Units']
df['CO2_per_Del']       = df['CO2_Saved_tons'] / (df['Estimated_Deliveries'] + 1)
df['Charging_per_Del']  = df['Charging_Stations'] / (df['Estimated_Deliveries'] + 1)
df['Log_Deliveries']    = np.log1p(df['Estimated_Deliveries'])

# ── Label encode categoricals
le_region = LabelEncoder()
le_model  = LabelEncoder()
le_source = LabelEncoder()
df['Region_Enc'] = le_region.fit_transform(df['Region'])
df['Model_Enc']  = le_model.fit_transform(df['Model'])
df['Source_Enc'] = le_source.fit_transform(df['Source_Type'])

print('Feature engineering complete. New shape:', df.shape)
print('New columns:', ['Date','Month_Sin','Month_Cos','Quarter','Years_Since_Start',
                       'Price_Per_km','Util_Rate','CO2_per_Del','Charging_per_Del',
                       'Log_Deliveries','Region_Enc','Model_Enc','Source_Enc'])

In [ ]:
# ── Visualise engineered features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['Price_Per_km'], bins=30, color='#4C72B0', edgecolor='white')
axes[0].set_title('Price per km (USD/km)')

axes[1].hist(df['Util_Rate'], bins=30, color='#55A868', edgecolor='white')
axes[1].set_title('Utilisation Rate (Deliveries/Production)')

axes[2].scatter(df['Years_Since_Start'], df['Avg_Price_USD'], alpha=0.3, s=10, color='#C44E52')
axes[2].set_title('Price over Years Since Launch')
axes[2].set_xlabel('Years Since 2015'); axes[2].set_ylabel('Avg Price (USD)')

plt.suptitle('Engineered Features Overview', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Regression Modeling — Predict `Avg_Price_USD`

In [ ]:
FEATURES = [
    'Year', 'Month', 'Quarter', 'Month_Sin', 'Month_Cos', 'Years_Since_Start',
    'Region_Enc', 'Model_Enc', 'Source_Enc',
    'Battery_Capacity_kWh', 'Range_km', 'Charging_Stations',
    'Util_Rate', 'Price_Per_km', 'Estimated_Deliveries'
]
TARGET = 'Avg_Price_USD'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    cv   = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2')
    return {
        'Model':       name,
        'MAE':         mean_absolute_error(y_te, pred),
        'RMSE':        np.sqrt(mean_squared_error(y_te, pred)),
        'R2 (test)':   r2_score(y_te, pred),
        'CV R2 Mean':  cv.mean(),
        'CV R2 Std':   cv.std(),
    }

models = {
    'Ridge':         Ridge(alpha=100),
    'Lasso':         Lasso(alpha=10),
    'RandomForest':  RandomForestRegressor(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1),
    'GradBoost':     GradientBoostingRegressor(n_estimators=150, random_state=RANDOM_STATE),
    'LightGBM':      lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=RANDOM_STATE, verbose=-1),
}

results = [evaluate(name, mdl, X_train, y_train, X_test, y_test) for name, mdl in models.items()]
results_df = pd.DataFrame(results).set_index('Model')
results_df.style.highlight_min(subset=['MAE','RMSE'], color='#d4f0d4') \
                .highlight_max(subset=['R2 (test)','CV R2 Mean'], color='#d4f0d4') \
                .format({'MAE': '{:.0f}', 'RMSE': '{:.0f}', 'R2 (test)': '{:.4f}',
                         'CV R2 Mean': '{:.4f}', 'CV R2 Std': '{:.4f}'})

In [ ]:
# ── Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

r2_vals = results_df['R2 (test)']
mae_vals = results_df['MAE']
colors = ['#C44E52' if v < r2_vals.max() else '#55A868' for v in r2_vals]

axes[0].barh(r2_vals.index, r2_vals.values, color=colors)
axes[0].set_xlabel('R² (test)'); axes[0].set_title('Model R² Comparison', fontweight='bold')
axes[0].set_xlim(0.85, 1.0)
for i, v in enumerate(r2_vals.values):
    axes[0].text(v + 0.0005, i, f'{v:.4f}', va='center', fontsize=9)

axes[1].barh(mae_vals.index, mae_vals.values, color=colors)
axes[1].set_xlabel('MAE (USD)'); axes[1].set_title('Model MAE Comparison', fontweight='bold')
for i, v in enumerate(mae_vals.values):
    axes[1].text(v + 10, i, f'${v:,.0f}', va='center', fontsize=9)

plt.tight_layout(); plt.show()
print(f'Best model: {r2_vals.idxmax()} (R²={r2_vals.max():.4f})')

## 6. Hyperparameter Tuning (LightGBM)

In [ ]:
param_grid = {
    'n_estimators':  [100, 200, 300],
    'learning_rate': [0.03, 0.05, 0.1],
    'num_leaves':    [31, 63],
}

gs = GridSearchCV(
    lgb.LGBMRegressor(random_state=RANDOM_STATE, verbose=-1),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)
gs.fit(X_train, y_train)

best_lgb = gs.best_estimator_
pred_best = best_lgb.predict(X_test)

print(f'Best params:  {gs.best_params_}')
print(f'Best CV R²:   {gs.best_score_:.4f}')
print(f'Test MAE:     ${mean_absolute_error(y_test, pred_best):,.0f}')
print(f'Test RMSE:    ${np.sqrt(mean_squared_error(y_test, pred_best)):,.0f}')
print(f'Test R²:      {r2_score(y_test, pred_best):.4f}')

In [ ]:
# ── Actual vs Predicted scatter
plt.figure(figsize=(8, 6))
plt.scatter(y_test, pred_best, alpha=0.4, s=15, color='steelblue', label='Predictions')
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
plt.xlabel('Actual Price (USD)'); plt.ylabel('Predicted Price (USD)')
plt.title('LightGBM (Tuned) — Actual vs Predicted', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Residual plot
residuals = y_test - pred_best
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(pred_best, residuals, alpha=0.4, s=15, color='darkorange')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted', fontweight='bold')

axes[1].hist(residuals, bins=40, color='#4C72B0', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual'); axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution', fontweight='bold')

plt.tight_layout(); plt.show()

In [ ]:
# ── Feature importance
fi = pd.Series(best_lgb.feature_importances_, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(9, 6))
colors_fi = ['#2166ac' if v == fi.max() else '#4393c3' if v >= fi.quantile(0.75) else '#92c5de'
             for v in fi.values]
plt.barh(fi.index, fi.values, color=colors_fi)
plt.xlabel('Importance Score')
plt.title('LightGBM Feature Importances', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 7. Time Series Forecasting — Global Monthly Deliveries

In [ ]:
# ── Aggregate to monthly global series
ts = df.groupby('Date')['Estimated_Deliveries'].sum().sort_index()
ts.index = pd.DatetimeIndex(ts.index)

plt.figure(figsize=(13, 4))
plt.plot(ts.index, ts.values, linewidth=1.2, color='steelblue')
plt.title('Monthly Global Tesla Deliveries (all regions, all models)', fontsize=13, fontweight='bold')
plt.ylabel('Total Deliveries')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
plt.tight_layout(); plt.show()

In [ ]:
# ── ACF / PACF plots to inspect stationarity and seasonality
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(ts, lags=36, ax=axes[0])
axes[0].set_title('Autocorrelation (ACF)', fontweight='bold')
plot_pacf(ts, lags=36, ax=axes[1], method='ywm')
axes[1].set_title('Partial Autocorrelation (PACF)', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Train / test split: train on <2024, test on 2024-2025
train_ts = ts[ts.index < '2024-01-01']
test_ts  = ts[ts.index >= '2024-01-01']
print(f'Train: {len(train_ts)} months ({train_ts.index.min().date()} → {train_ts.index.max().date()})')
print(f'Test:  {len(test_ts)}  months ({test_ts.index.min().date()} → {test_ts.index.max().date()})')

In [ ]:
# ── Holt-Winters Triple Exponential Smoothing
hw_model = ExponentialSmoothing(
    train_ts,
    trend='add',
    seasonal='add',
    seasonal_periods=12
).fit(optimized=True)

hw_pred = hw_model.forecast(len(test_ts))

ts_mae  = mean_absolute_error(test_ts, hw_pred)
ts_rmse = np.sqrt(mean_squared_error(test_ts, hw_pred))
ts_mape = (np.abs((test_ts.values - hw_pred.values) / test_ts.values)).mean() * 100

print(f'Holt-Winters Forecast Results')
print(f'  MAE:  {ts_mae:,.0f}')
print(f'  RMSE: {ts_rmse:,.0f}')
print(f'  MAPE: {ts_mape:.2f}%')

In [ ]:
# ── Forecast plot
plt.figure(figsize=(13, 5))

plt.plot(train_ts.index[-36:], train_ts.values[-36:],
         color='steelblue', linewidth=1.5, label='Train (last 3 years)')
plt.plot(test_ts.index, test_ts.values,
         color='#2ca02c', linewidth=1.8, label='Actual (test)')
plt.plot(hw_pred.index, hw_pred.values,
         color='#d62728', linewidth=1.8, linestyle='--', label='Holt-Winters Forecast')

# confidence band (±1 RMSE)
plt.fill_between(hw_pred.index,
                 hw_pred.values - ts_rmse,
                 hw_pred.values + ts_rmse,
                 alpha=0.15, color='#d62728', label='±1 RMSE band')

plt.axvline(pd.Timestamp('2024-01-01'), color='gray', linestyle=':', linewidth=1)
plt.text(pd.Timestamp('2024-01-01'), ts.max() * 0.97, ' ← Test', color='gray', fontsize=9)
plt.title('Holt-Winters Triple Exponential Smoothing Forecast', fontsize=13, fontweight='bold')
plt.ylabel('Monthly Global Deliveries')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Forecast error analysis
error_df = pd.DataFrame({
    'Actual':    test_ts.values,
    'Forecast':  hw_pred.values,
    'Error':     test_ts.values - hw_pred.values,
    'Pct_Error': (test_ts.values - hw_pred.values) / test_ts.values * 100
}, index=test_ts.index)

print(error_df.head(12).to_string(float_format='{:.1f}'.format))

In [ ]:
# ── Future forecast: 12 months beyond dataset
future_hw = ExponentialSmoothing(
    ts, trend='add', seasonal='add', seasonal_periods=12
).fit(optimized=True)

future_pred = future_hw.forecast(12)

plt.figure(figsize=(13, 5))
plt.plot(ts.index[-24:], ts.values[-24:], color='steelblue', linewidth=1.5, label='Historical')
plt.plot(future_pred.index, future_pred.values, color='darkorange', linewidth=2,
         linestyle='--', label='12-Month Forecast')
plt.fill_between(future_pred.index,
                 future_pred.values * 0.92,
                 future_pred.values * 1.08,
                 alpha=0.2, color='darkorange', label='±8% uncertainty band')
plt.title('12-Month Future Delivery Forecast (beyond 2025)', fontsize=13, fontweight='bold')
plt.ylabel('Monthly Deliveries')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
plt.legend()
plt.tight_layout(); plt.show()

print('\nForecast summary:')
print(future_pred.rename('Predicted_Deliveries').to_frame().to_string())

## 8. Pipeline Summary

In [ ]:
summary = pd.DataFrame([
    ['Dataset',             'tesla_deliveries_dataset_2015_2025.csv', '2,640 rows × 12 cols'],
    ['Missing Values',      'None',                                   '—'],
    ['Feature Engineering', '13 new features',                        'cyclical, ratio, log, label-enc'],
    ['Best Reg. Model',     'LightGBM (tuned)',                       f'R²={r2_score(y_test, pred_best):.4f}, MAE=${mean_absolute_error(y_test, pred_best):,.0f}'],
    ['TS Forecaster',       'Holt-Winters (add+add)',                  f'MAE={ts_mae:,.0f}, MAPE={ts_mape:.2f}%'],
    ['Top Feature',         'Price_Per_km',                           'Highest LightGBM importance'],
], columns=['Stage', 'Result', 'Notes'])

summary.style.set_properties(**{'text-align': 'left'}).hide(axis='index')